# 🫁 Optimized Hybrid CNN+Swin Research Pipeline (BCE Loss & 384 Resolution)

This notebook implements the final research pipeline with the following improvements:
1. **Architecture**: DenseNet121 + Swin Transformer (Tiny variant with 384 adaptation) + Spatial Attention.
2. **Loss**: BCEWithLogitsLoss (Stable Baseline).
3. **Resolution**: 384x384 with CLAHE Preprocessing.
4. **Split**: Patient-wise to ensure zero data leakage.
5. **Logging**: Per-label metrics (AUC) saved to CSV after every best epoch.

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, f1_score
from PIL import Image
from tqdm.auto import tqdm
import torch.nn.functional as F

# --- CONFIGURATION ---
BASE_DIR = os.path.dirname(os.getcwd()) 
DATA_DIR = os.path.join(BASE_DIR, 'data', 'images')
METADATA_PATH = os.path.join(BASE_DIR, 'data', 'metadata_filtered.csv')
RAW_METADATA_PATH = os.path.join(BASE_DIR, 'data', 'Data_Entry_2017_v2020.csv')

EXPERIMENT_DIR = os.getcwd() 
MODEL_SAVE_PATH = os.path.join(EXPERIMENT_DIR, 'hybrid_384_bce_best.pth')
RESULTS_CSV_PATH = os.path.join(EXPERIMENT_DIR, 'experiment_results.csv')

TARGET_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 
    'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
IMG_SIZE = 384
BATCH_SIZE = 8
EPOCHS = 10
NUM_WORKERS = 0 # Safety for macOS spawn issues

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Device: {device} | Experiment Folder: {EXPERIMENT_DIR}")

## 1. Model Architecture

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        res = torch.cat([avg_out, max_out], dim=1)
        res = self.conv(res)
        return x * self.sigmoid(res)

class HybridResearchModel(nn.Module):
    def __init__(self, num_classes=len(TARGET_CLASSES)):
        super().__init__()
        cnn = models.densenet121(weights='IMAGENET1K_V1')
        self.cnn_features = cnn.features
        self.cnn_pool = nn.AdaptiveAvgPool2d(1)
        
        # Using swin_tiny_patch4_window7_224 and adapting it to 384 pixels
        self.vit = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0, img_size=IMG_SIZE)
        
        self.spatial_att = SpatialAttention()
        self.classifier = nn.Sequential(
            nn.Linear(1024 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.final_fc = nn.Linear(512, num_classes)

    def forward(self, x):
        c_feat = self.cnn_features(x)
        c_feat = self.spatial_att(c_feat)
        c_feat = self.cnn_pool(c_feat).view(x.size(0), -1) 
        v_feat = self.vit(x)
        combined = torch.cat([c_feat, v_feat], dim=1)
        out = self.classifier(combined)
        return self.final_fc(out)

## 2. Dataset and Preprocessing

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        try:
            img_name = self.dataframe.iloc[idx]['Image Index']
            img_path = os.path.join(self.root_dir, img_name)
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None: return torch.zeros((3, IMG_SIZE, IMG_SIZE)), torch.zeros(len(TARGET_CLASSES))
            image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
            image = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(image)
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            image = Image.fromarray(image)
            label = torch.tensor(self.dataframe.iloc[idx]['label_vec'], dtype=torch.float32)
            if self.transform: image = self.transform(image)
            return image, label
        except: return torch.zeros((3, IMG_SIZE, IMG_SIZE)), torch.zeros(len(TARGET_CLASSES))

def prepare_loaders():
    df = pd.read_csv(METADATA_PATH)
    if 'Patient ID' not in df.columns:
        raw_df = pd.read_csv(RAW_METADATA_PATH, usecols=['Image Index', 'Patient ID'])
        df = df.merge(raw_df, on='Image Index', how='left')
    def encode(l): return [1 if c in str(l).split('|') else 0 for c in TARGET_CLASSES]
    df['label_vec'] = df['Finding Labels'].apply(encode)
    imgs_on_disk = set(os.listdir(DATA_DIR))
    df = df[df['Image Index'].isin(imgs_on_disk)].reset_index(drop=True)
    unique_patients = df['Patient ID'].unique()
    np.random.seed(42)
    np.random.shuffle(unique_patients)
    train_p = unique_patients[:int(0.8*len(unique_patients))]
    val_p = unique_patients[int(0.8*len(unique_patients)):int(0.9*len(unique_patients))]
    train_df = df[df['Patient ID'].isin(train_p)].reset_index(drop=True)
    val_df = df[df['Patient ID'].isin(val_p)].reset_index(drop=True)
    train_transform = transforms.Compose([
        transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    train_loader = DataLoader(ChestXrayDataset(train_df, DATA_DIR, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(ChestXrayDataset(val_df, DATA_DIR, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, val_loader

## 3. Training Loop

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Validating"):
            out = torch.sigmoid(model(imgs.to(device)))
            all_preds.append(out.cpu().numpy())
            all_labels.append(labels.numpy())
    y_true = np.vstack(all_labels)
    y_pred = np.vstack(all_preds)
    results = []
    for i, cls in enumerate(TARGET_CLASSES):
        try:
            auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            results.append({'Class': cls, 'AUC': auc})
        except:
            results.append({'Class': cls, 'AUC': 0})
    macro_auc = roc_auc_score(y_true, y_pred, average='macro')
    return pd.DataFrame(results), macro_auc

train_loader, val_loader = prepare_loaders()
model = HybridResearchModel().to(device)
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_auc = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'Loss': f"{loss.item():.6f}"})
    res_df, m_auc = evaluate(model, val_loader, device)
    print(f"\n✅ Epoch {epoch} Macro AUC: {m_auc:.4f}")
    if m_auc > best_auc:
        best_auc = m_auc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        res_df.to_csv(RESULTS_CSV_PATH, index=False)
        print(f"⭐ Best Model and Results saved to {RESULTS_CSV_PATH}")
    scheduler.step()
    if device.type == 'mps': torch.backends.mps.empty_cache()